# TAE-IA · Módulo 6 · L21 — Speech Synthesis: Three Ways to Make a Voice

| | |
|---|---|
| **Engines** | gTTS (cloud) · Piper `es_MX-claude-high` 63 MB (CPU) · XTTS-v2 1.9 GB (GPU) |
| **Also loads** | Whisper `small` 461 MB, for the round trip |
| **Runtime** | T4 GPU · ~2.5 GB on the runtime disk, re-downloaded each session |
| **Outputs** | `TAE_IA_M6/L21_output/` on Drive — a few MB of WAV files |

## Learning objectives

1. Synthesise the same Spanish text with three architecturally different engines and hear the difference
2. Move each engine's controls and connect every knob to the model part it scales
3. Measure intelligibility (round-trip WER through Whisper) and speed (real-time factor) honestly
4. Run a blind listening test and compare what the room hears with what the metrics say

## 1 — Setup

Colab does not ship `coqui-tts`, `piper-tts` or `gtts`. The `[codec]` extra pulls in `torchcodec`,
which `torchaudio` needs for audio I/O from torch 2.9 on. About two minutes.

In [ ]:
!pip install -q "coqui-tts[codec]" gtts piper-tts jiwer openai-whisper

**Two warnings you can ignore today:**
- pip's red *"dependency resolver … diffusers requires huggingface-hub>=1.23"* (and `click` for
  wandb/spacy). `coqui-tts` pins an older `huggingface_hub`; nothing in this lab uses diffusers, wandb
  or spacy. It *would* matter in a notebook that also loads a diffusers pipeline.
- `[transformers] Model config: bos_token_id must be None or an integer within the vocabulary…` when
  XTTS loads. XTTS's GPT uses its own audio-code vocabulary; the check is written for text models.

In [ ]:
import os, re, time, random, wave
import numpy as np
import torch

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/TAE_IA_M6'
OUTPUT_DIR = f'{DRIVE_ROOT}/L21_output'
INPUT_DIR  = f'{DRIVE_ROOT}/inputs'
CLIPS      = f'{OUTPUT_DIR}/clips'
for d in (OUTPUT_DIR, INPUT_DIR, CLIPS):
    os.makedirs(d, exist_ok=True)

# Weights on the runtime disk: fast, no Drive symlink bug, wiped between sessions.
MODEL_CACHE = '/content/models'
os.makedirs(MODEL_CACHE, exist_ok=True)
os.environ['HF_HOME']  = MODEL_CACHE
os.environ['TTS_HOME'] = f'{MODEL_CACHE}/tts'
os.environ['COQUI_TOS_AGREED'] = '1'   # XTTS-v2: Coqui Public Model License, non-commercial use only

if not torch.cuda.is_available():
    raise SystemExit('No GPU. Runtime > Change runtime type > T4 GPU')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def vram(tag=''):
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  VRAM {used:5.2f} / {total:.1f} GB   {tag}')

print(torch.cuda.get_device_name(0)); vram('empty')

In [ ]:
# coqui-tts 0.27 still imports a helper that transformers 5.1 removed
# (idiap/coqui-ai-TTS issue #558). Put it back BEFORE importing TTS.
import transformers, transformers.pytorch_utils as pu
if not hasattr(pu, 'isin_mps_friendly'):
    pu.isin_mps_friendly = torch.isin
    print(f'transformers {transformers.__version__}: compatibility alias added')

import soundfile as sf, librosa, librosa.display
import matplotlib.pyplot as plt
import IPython.display as ipd
from gtts import gTTS
from piper import PiperVoice, SynthesisConfig
from huggingface_hub import hf_hub_download
from TTS.api import TTS
%matplotlib inline
# ^ Importing TTS switches matplotlib to the non-interactive 'Agg' backend
#   (TTS/tts/utils/visual.py calls matplotlib.use("Agg")), which silently hides every plot.
#   This line switches it back. Without it, Parts 2 and 3 show no figures.

plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

def seconds(path):
    info = sf.info(path)
    return info.frames / info.samplerate

def play(path, label=''):
    print(f'{label:28s} {seconds(path):5.2f} s   {os.path.basename(path)}')
    ipd.display(ipd.Audio(path))

## 2 — Three engines behind one interface

Each `synth_*` function takes text and a clip name, writes a WAV to `CLIPS`, and returns
`(path, synthesis_seconds)`. Same signature for all three, so everything after this cell treats
them identically.

- **gTTS** sends the text to Google Translate's TTS service. `tld` picks the accent.
- **Piper** runs a VITS voice on the CPU through ONNX Runtime.
- **XTTS-v2** runs a 443M-parameter GPT-2 plus a HiFi-GAN decoder on the GPU.

In [ ]:
def synth_gtts(text, name, tld='com.mx'):
    mp3, wav = f'{CLIPS}/{name}.mp3', f'{CLIPS}/{name}.wav'
    t0 = time.time()
    gTTS(text=text, lang='es', tld=tld).save(mp3)
    dt = time.time() - t0                          # network + synthesis, the user's view
    y, sr = librosa.load(mp3, sr=None)             # MP3 -> WAV so every engine is measured the same way
    sf.write(wav, y, sr)
    return wav, dt

PIPER_STEM = 'es/es_MX/claude/high/es_MX-claude-high'
piper_onnx = hf_hub_download('rhasspy/piper-voices', PIPER_STEM + '.onnx', local_dir=f'{MODEL_CACHE}/piper')
hf_hub_download('rhasspy/piper-voices', PIPER_STEM + '.onnx.json', local_dir=f'{MODEL_CACHE}/piper')
piper_voice = PiperVoice.load(piper_onnx)
print('Piper sample rate:', piper_voice.config.sample_rate)

def synth_piper(text, name, **knobs):              # knobs: length_scale, noise_scale, noise_w_scale
    wav = f'{CLIPS}/{name}.wav'
    t0 = time.time()
    with wave.open(wav, 'wb') as wf:
        piper_voice.synthesize_wav(text, wf, syn_config=SynthesisConfig(**knobs))
    return wav, time.time() - t0

t0 = time.time()
xtts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to('cuda')
print(f'XTTS-v2 loaded in {time.time()-t0:.0f} s'); vram('XTTS-v2 resident')
print(f'{len(xtts.speakers)} built-in speakers, e.g. {xtts.speakers[:6]}')
XTTS_SPEAKER = xtts.speakers[0]

def synth_xtts(text, name, speaker=None, speaker_wav=None, **gen):   # gen: temperature, top_p, speed, ...
    wav = f'{CLIPS}/{name}.wav'
    if speaker_wav is None and speaker is None:
        speaker = XTTS_SPEAKER
    torch.cuda.synchronize(); t0 = time.time()
    xtts.tts_to_file(text=text, language='es', file_path=wav,
                     speaker=speaker, speaker_wav=speaker_wav, **gen)
    torch.cuda.synchronize()
    return wav, time.time() - t0

ENGINES = {'gtts': synth_gtts, 'piper': synth_piper, 'xtts': synth_xtts}

## 3 — Part 1: the same sentences, three voices

Four sentences, each chosen to stress one stage of the pipeline:

| Key | Stresses |
|---|---|
| `probe` | the front-end: an abbreviation, a number and a date in one line |
| `fourier` | grapheme-to-phoneme on a French loanword |
| `question` | prosody: the rising pitch of a question |
| `long` | the stop decision and long-range rhythm |

In [ ]:
SENTENCES = {
    'probe':    'El Dr. Pérez midió 128 bandas el 15/09/2026.',
    'fourier':  'La transformada de Fourier revela las componentes de frecuencia de una señal.',
    'question': '¿Ya terminaste de entrenar el modelo, o todavía te falta la última época?',
    'long':     ('Los sistemas de síntesis de voz trabajan en tres etapas: normalizan el texto, '
                 'deciden la prosodia y reconstruyen la forma de onda con un vocoder.'),
}

clips = {}                                  # (engine, sentence) -> (path, synthesis seconds)
for key, text in SENTENCES.items():
    print(f'\n=== {key}: {text}')
    for eng, fn in ENGINES.items():
        clips[(eng, key)] = fn(text, f'p1_{eng}_{key}')
        play(clips[(eng, key)][0], eng)

**Listen before you measure.** For each sentence, write one line per engine: did it read *Dr.*,
*128* and the date correctly? Did *Fourier* come out French, Spanish-spelled, or broken? Did the
question actually rise at the end?

*Your notes:*

### XTTS clones — one sentence, two voices

XTTS takes a **reference clip** instead of a built-in speaker. Here the reference is gTTS's own
Mexican voice: XTTS imitates a voice that is itself synthetic. Cloning real people — and whether
you should — is L22.

In [ ]:
ref_wav, _ = synth_gtts('Hola. Ésta es la voz de referencia que XTTS va a imitar en los siguientes segundos.',
                        'ref_gtts_mx')
clone_wav, _ = synth_xtts(SENTENCES['question'], 'p1_xtts_clone_gtts', speaker_wav=ref_wav)
play(ref_wav, 'reference (gTTS com.mx)')
play(clips[('xtts', 'question')][0], f'XTTS built-in: {XTTS_SPEAKER}')
play(clone_wav, 'XTTS cloning the reference')

## 4 — Part 2: every knob maps to a part of the model

**Piper** exposes VITS's three noise knobs. **XTTS** exposes the GPT's sampling parameters.
Predict before you run: which Piper knob changes the *length* of the clip, and which only changes
how it *sounds*?

In [ ]:
TEXT = SENTENCES['question']
variants = {}
for ls in (0.8, 1.0, 1.4):
    variants[f'piper length_scale={ls}'] = synth_piper(TEXT, f'p2_piper_ls{ls}', length_scale=ls)[0]
for ns in (0.0, 0.667, 1.2):
    variants[f'piper noise_scale={ns}'] = synth_piper(TEXT, f'p2_piper_ns{ns}', noise_scale=ns)[0]
for temp in (0.3, 0.75, 1.1):
    torch.manual_seed(SEED)                      # same seed: only the temperature differs
    variants[f'xtts temperature={temp}'] = synth_xtts(TEXT, f'p2_xtts_t{temp}', temperature=temp)[0]
for sp in (0.8, 1.25):
    torch.manual_seed(SEED)
    variants[f'xtts speed={sp}'] = synth_xtts(TEXT, f'p2_xtts_sp{sp}', speed=sp)[0]

for label, path in variants.items():
    play(path, label)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={'width_ratios': [1.1, 1]})
labels = list(variants)
ax1.barh(labels, [seconds(variants[k]) for k in labels], color=['#27ae60'] * 6 + ['#1a2e5a'] * 5)
ax1.invert_yaxis(); ax1.set_xlabel('clip duration (s)'); ax1.set_title('Which knobs change the length?')

for k, c in zip(['piper noise_scale=0.0', 'piper noise_scale=1.2'], ['#1a2e5a', '#e67e22']):
    y, sr = librosa.load(variants[k], sr=None)
    f0, voiced, _ = librosa.pyin(y, fmin=70, fmax=400, sr=sr)
    t = librosa.times_like(f0, sr=sr)
    ax2.plot(t, f0, color=c, label=k)
ax2.set_xlabel('time (s)'); ax2.set_ylabel('F0 (Hz)'); ax2.legend(); ax2.set_title('Pitch contour: noise_scale 0 vs 1.2')
plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/p2_knobs.png'); plt.show()

**Observation.** Which knobs changed the duration, and by exactly the factor you set — or less? Did
`noise_scale` change the pitch contour at all? If not, where in VITS is the melody decided, and what
*does* the noise change when you listen? At what XTTS temperature did the first real mistake appear?

*Your answer:*

## 5 — Part 3: the round trip — WER and real-time factor

Whisper transcribes every clip from Part 1 and we score it against the text we wrote.

**Two measurement rules from the slides:**
1. **Normalise both sides the same way** — Whisper writes `128`, the reference may say *ciento
   veintiocho*. `normalise_es` expands numbers, the date and `Dr.` on *both* strings.
2. **Warm up before timing** (L06). The first call to each engine pays one-off costs.

In [ ]:
import whisper
from jiwer import wer
from num2words import num2words

asr = whisper.load_model('small', download_root=f'{MODEL_CACHE}/whisper')
vram('XTTS + Whisper small')

MESES = ['', 'enero', 'febrero', 'marzo', 'abril', 'mayo', 'junio', 'julio', 'agosto',
         'septiembre', 'octubre', 'noviembre', 'diciembre']

def normalise_es(text):
    t = text.lower()
    t = re.sub(r'(\d{1,2})/(\d{1,2})/(\d{4})',
               lambda m: f"{num2words(int(m[1]), lang='es')} de {MESES[int(m[2])]} de {num2words(int(m[3]), lang='es')}", t)
    t = re.sub(r'\bdr\.?\s', 'doctor ', t)
    t = re.sub(r'\d+', lambda m: num2words(int(m[0]), lang='es'), t)
    t = re.sub(r'[^a-záéíóúüñ\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

print(normalise_es(SENTENCES['probe']))
print(normalise_es('El doctor Pérez midió 128 bandas el 15 de septiembre de 2026.'))

In [ ]:
import pandas as pd

for eng, fn in ENGINES.items():                     # warm-up: one short call each, not timed
    fn('Prueba.', f'warmup_{eng}')

rows = []
for key, text in SENTENCES.items():
    for eng, fn in ENGINES.items():
        path, dt = fn(text, f'p3_{eng}_{key}')
        hyp = asr.transcribe(path, language='es', fp16=True)['text'].strip()
        rows.append({'engine': eng, 'sentence': key, 'synth_s': dt, 'audio_s': seconds(path),
                     'RTF': dt / seconds(path), 'WER': wer(normalise_es(text), normalise_es(hyp)),
                     'heard': hyp})

df = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', 90)
display(df.round(3))
summary = df.groupby('engine')[['RTF', 'WER']].mean().round(3)
display(summary)
df.to_csv(f'{OUTPUT_DIR}/p3_roundtrip.csv', index=False)

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
summary['RTF'].plot.bar(ax=a1, color=['#27ae60', '#1a2e5a', '#e67e22'], logy=True)
a1.axhline(1, ls='--', c='grey'); a1.set_title('Real-time factor (log scale, < 1 is faster than real time)')
summary['WER'].plot.bar(ax=a2, color=['#27ae60', '#1a2e5a', '#e67e22'])
a2.set_title('Round-trip WER (normalised)')
for a in (a1, a2): a.tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/p3_rtf_wer.png'); plt.show()

# The same XTTS call twice, no fixed seed: does the transcript change, or only the sound?
torch.manual_seed(int(time.time()))
twice = [synth_xtts(SENTENCES['long'], f'p3_xtts_twice_{i}')[0] for i in (1, 2)]
for p in twice:
    play(p, 'XTTS, unseeded')
    print('   heard:', asr.transcribe(p, language='es', fp16=True)['text'].strip())

**Observation.** Fill in the RTF cell for XTTS in your copy of the *Three Engines, One Table*
slide. Which engine had the highest WER, and on which sentence? **Listen to every clip whose WER is
not 0** and decide: engine error, normalisation mismatch, or Whisper's mistake? Did the two unseeded
XTTS runs transcribe identically?

*Your answer:*

## 6 — Part 4: a blind listening test

Six clips, shuffled, labels hidden. Rate each one **1 (bad) to 5 (excellent)** for naturalness
*before* revealing which engine made it. Better still: swap laptops with your neighbour and rate
theirs.

In [ ]:
rng = random.Random()                               # deliberately unseeded: a different order per student
pool = [(eng, key) for key in ('question', 'long') for eng in ENGINES]
rng.shuffle(pool)
LETTERS = 'ABCDEF'
blind = dict(zip(LETTERS, pool))
for letter, (eng, key) in blind.items():
    print(f'Clip {letter}')
    ipd.display(ipd.Audio(clips[(eng, key)][0]))

In [ ]:
# Rate every clip 1-5 BEFORE running the reveal cell.
RATINGS = {'A': None, 'B': None, 'C': None, 'D': None, 'E': None, 'F': None}

In [ ]:
missing = [k for k, v in RATINGS.items() if v is None]
if missing:
    print(f'Rate clips {missing} first.')
else:
    reveal = pd.DataFrame([{'clip': k, 'engine': blind[k][0], 'sentence': blind[k][1], 'rating': v}
                           for k, v in RATINGS.items()])
    display(reveal)
    mos = reveal.groupby('engine')['rating'].agg(['mean', 'count']).round(2)
    display(mos.join(summary))
    reveal.to_csv(f'{OUTPUT_DIR}/p4_blind_ratings.csv', index=False)

**Observation.** Did your naturalness ranking match the WER ranking from Part 3? With two ratings
per engine from one listener, what can this "MOS" honestly claim — and what would it take to make it
a real one? When the instructor collects the room's scores, does your ranking agree with the room's?

*Your answer:*

## Exercise 1 — Break the front-end

Write **five Spanish sentences** that each attack the text front-end in a different way — pick from:
an ordinal (*3er*), money (*$1,250.50*), a time (*14:30 h*), an acronym (*UNAM*, *CFE*), an English
brand (*YouTube*, *Wi-Fi*), a unit (*5 km/h*), a phone number, Roman numerals (*siglo XXI*).

For every sentence × engine: synthesise, run the round trip, **listen**, and label each error as
**front-end** (read wrong), **engine** (pronounced badly or skipped) or **ASR** (Whisper's mistake).

In [ ]:
# Exercise 1 -- Break the front-end
# Given: ENGINES, asr, normalise_es, wer, play, pd are all defined above.

MY_SENTENCES = [
    # TODO 1: five sentences, each attacking the front-end differently
]

# TODO 2: for each sentence and each engine, synthesise with a unique clip name ('ex1_...'),
#         transcribe with asr (language='es', fp16=True), and compute WER on normalised text.
# TODO 3: collect the results in a DataFrame and display it.
# TODO 4: listen to every clip with WER > 0 and write your error labels in the cell below.

**Exercise 1 — Answer:** a table of your error labels, then: which engine's front-end was the most
robust, and which error type dominated overall? Name one error that `normalise_es` itself
introduced or hid.

*Your answer:*

## Exercise 2 — Does the real-time factor stay constant?

Measure RTF for **Piper and XTTS** on texts of roughly **5, 20, 60 and 150 words**, built from
whole sentences so they keep their full stops. Plot RTF against word count for both engines.

Is RTF constant as the text grows? Then the XTTS experiment: `tts_to_file` **splits text into
sentences by default** and synthesises them one at a time. Run the 150-word text again with
`split_sentences=False` — one pass, far past XTTS's 239-character limit for Spanish — and compare
time and sound. Read the warning coqui-tts prints.

In [ ]:
# Exercise 2 -- RTF vs. text length
# Given: synth_piper, synth_xtts, seconds, plt

SHORT = '¿Ya terminaste?'          # ~2 words
LONG  = SENTENCES['long']          # ~22 words, one full stop

# TODO 1: build four texts of about 5, 20, 60 and 150 words from SHORT and repetitions of LONG
#         (join whole sentences with spaces so every copy keeps its full stop)
# TODO 2: for each text, time synth_piper and synth_xtts (unique clip names 'ex2_...') and compute RTF
# TODO 3: plot RTF against word count, one line per engine
# TODO 4: synthesise the 150-word text with synth_xtts(..., split_sentences=False) and compare
#         time and sound with the default (split) clip from TODO 2

**Exercise 2 — Answer:** your plot, then: is RTF constant for each engine, and why (think about
what runs once per call versus once per frame)? What happened to the one-pass 150-word XTTS clip,
and what does sentence splitting trade to avoid it?

*Your answer:*

## Part 4 — Critical Analysis

### Q1 — Choose an engine, twice

Scenario A: an information kiosk in a museum in Oaxaca, no reliable internet, CPU only. Scenario B: an audiobook narrated in the voice of a consenting narrator, rendered overnight on a GPU. Pick an engine for each and justify it with **rows of your measured table**, not with the slide.

*Your answer:*

### Q2 — The prediction you got wrong

From the Demo Checkpoint, take the prediction that was furthest from your result. Explain the mechanism behind what actually happened, referring to one slide's model diagram.

*Your answer:*

### Q3 — When WER and your ears disagreed

Find one clip in your own results where the round-trip WER and your listening disagreed. Say which one was right and why the other was misled.

*Your answer:*

### Q4 — Whose Spanish?

Your final project will ship a default Spanish voice. Which one would you choose, for which users, and what would you tell users who do not sound like it?

*Your answer:*

## Submission Checklist

- [ ] Part 1 clips listened to, with notes per engine
- [ ] `p2_knobs.png`, `p3_rtf_wer.png`, `p3_roundtrip.csv` and `p4_blind_ratings.csv` in `L21_output/`
- [ ] Observation cells in Parts 2, 3 and 4 answered
- [ ] Exercises 1 and 2 completed with their answer cells
- [ ] Critical Analysis Q1–Q4 answered with **your** numbers

## Before You Close This Tab

Your clips and tables are on Drive. The 2.5 GB of weights are on the runtime disk and vanish with
it. **Runtime → Disconnect and delete runtime** — an idle GPU session still burns quota.

**Tonight, for L22:** record your voice — instructions and the text to read are in the next cell.

## Tonight — record your voice for L22 (10 minutes)

Tomorrow you train a voice-conversion model (RVC) **on your own voice** and make it sing. The model
is only as good as this recording.

**How**
- Your phone's voice recorder — WAV or M4A (not a WhatsApp voice note: too compressed)
- A quiet, soft room: no TV, music or fan. A closet full of clothes beats a kitchen
- Phone 15–20 cm from your mouth, and keep it there
- Normal voice — no whispering, no shouting. If playback distorts, move back
- Only your voice: nobody talking in the background

**What — 3 to 5 minutes in total**
1. Read the text below calmly, as if telling someone (~3 min)
2. Sing 30–60 seconds of anything you know, even badly — include a low part and a high part

**Upload** to `TAE_IA_M6/inputs/voz/` as `voz_<tu_nombre>.m4a` (or `.wav`). You can use
`upload_inputs()`-style Colab upload or drag it into Drive from your phone.

**Not comfortable cloning your voice?** That is a legitimate choice. Tell the instructor and you will
use a volunteer's voice from an openly licensed Spanish speech dataset, recorded for speech technology. Your recording and your model stay on your Drive.

---

### Texto para leer

> El sonido es una vibración que viaja por el aire. Cuando hablamos, los pulmones empujan el aire,
> las cuerdas vocales vibran y la boca le da forma a cada vocal y a cada consonante. Por eso una
> computadora puede aprender a reconocer una voz: cada persona tiene un tracto vocal distinto.
>
> ¿Alguna vez has escuchado tu voz grabada y pensado que no suenas así? Es normal. Dentro de tu
> cabeza escuchas tu voz también a través de los huesos, y eso la hace sonar más grave.
>
> Ayer, a las siete y media, el perro del vecino ladró durante veintitrés minutos. Jorge, que
> vive en el tercer piso, bajó corriendo con un paraguas rojo, una linterna y un chaleco amarillo.
> "¡Ya basta!", gritó, pero el perro siguió ladrando hasta que llegó la lluvia.
>
> En la ciudad de Guadalajara hay mercados llenos de colores, olores y ruidos: el zumbido de los
> refrigeradores, el choque de las ollas, la gente que ofrece jugo de naranja, churros y elotes.
> Si cierras los ojos, puedes adivinar dónde estás solo por lo que oyes.
>
> Una red neuronal no escucha como nosotros. Convierte el audio en números, calcula un espectrograma
> y busca patrones. Con suficientes ejemplos aprende que la erre fuerte de "ferrocarril" no se parece
> a la ele de "lluvia", ni la eñe de "mañana" a la jota de "Jalisco".
>
> Hoy la inteligencia artificial puede imitar una voz con pocos minutos de grabación. Eso abre
> puertas maravillosas, como devolverle la voz a alguien que la perdió, pero también riesgos
> serios, como el fraude telefónico. Por eso la regla de este curso es simple: solo se clona una
> voz con el permiso de su dueño. Hoy, el dueño eres tú.
>
> Uno, dos, tres, cuatro, cinco, seis, siete, ocho, nueve, diez. Mil novecientos ochenta y cinco.
> Doce de diciembre. ¿Cómo estás? ¡Qué gusto verte! Bueno, nos vemos mañana.